<a href="https://colab.research.google.com/github/BraedynL0530/CanopyWatch/blob/master/deforestation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#network stuff here when i leave collab

In [ ]:
import io
import requests
import rasterio
import numpy as np

#DO NOT RUN LOCALLY MY PC WILL EXPLODE!!!!!!!!!!
fastapi_url = "http://127.0.0.1:8000/api/get-satellite-patch" #wont work in google collab(unless i tunnell) this is for deploy/temp link
print("Fetching data from server...")
response = requests.get(fastapi_url)

if response.status_code == 204:
    print("No large NVDI change")
    exit()
elif response.status_code != 200:
    print(f"Error fetching: {response.status_code}")
    exit()

tiff_bytes = response.content

with rasterio.open(io.BytesIO(tiff_bytes)) as src:
    print("--- Raster Metadata ---")
    print(f"Width x height: {src.width}x{src.height}")
    print(f"Band Count: {src.count}")
    print(f"Coordinate System: {src.crs}")
    print(f"Geotransform Matrix:\n{src.transform}")
    print("-----------------------")

    #Shape: (4, 512, 512)  (Bands, Height, Width)
    full_stack = src.read()

    #slice the first 3 bands out for visual evidence / frontend
    rgb_channels = full_stack[0:3, :, :]  # Shape: (3, 512, 512)


    nir_channel = full_stack[3, :, :]     # Shape: (512, 512)

print("!!unpacked channels!!")

In [ ]:
#model
import torch
import torch.nn as nn

class forestClassifier(nn.Module):
    def __init__(self):
        super(forestClassifier, self).__init__()
        self.encoder = nn.Conv2d(4, 64, kernel_size=3,  padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneckConv = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.upsample = nn.ConvTranspose2d(128,64,kernel_size=2,stride=2)

        self.finalConv = nn.Conv2d(64,1,kernel_size=1)

        self.relu = nn.ReLU()
    def forward(self, x):
        x1 = self.relu(self.encoder(x))
        x2 = self.pool(x1)
        b=self.relu(self.bottleneckConv(x2))
        u=self.relu(self.upsample(b))
        out = self.finalConv(u)
        return out



In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter
import rasterio
from rasterio.enums import Resampling
import os

class DeforestationDataset(Dataset):
    def __init__(self, root_dir):
        self.image_paths = []
        self.mask_paths = []

        img_dir = os.path.join(root_dir, "1_CLOUD_FREE_DATASET", "2_SENTINEL2")
        mask_dir = os.path.join(root_dir, "3_TRAINING_MASKS")

        if os.path.exists(img_dir) and os.path.exists(mask_dir):
            for root, _, files in os.walk(img_dir):
                for file in files:
                    if file.lower().endswith(('.tif', '.png', '.jpg')):
                        self.image_paths.append(os.path.join(root, file))

            for root, _, files in os.walk(mask_dir):
                for file in files:
                    if file.lower().endswith(('.tif', '.png', '.jpg')):
                        self.mask_paths.append(os.path.join(root, file))

            self.image_paths.sort()
            self.mask_paths.sort()
        else:
            print(f"folder structure not found in {root_dir}")

        if not self.image_paths or len(self.image_paths) != len(self.mask_paths):
            print("extracted weird or smth")

    def __len__(self):
        return min(len(self.image_paths), len(self.mask_paths))

    def __getitem__(self, idx):
        with rasterio.open(self.image_paths[idx]) as src:
            img_np = src.read(
                out_shape=(src.count, 512, 512),
                resampling=Resampling.bilinear
            )

        with rasterio.open(self.mask_paths[idx]) as src:
            mask_np = src.read(
                out_shape=(src.count, 512, 512),
                resampling=Resampling.nearest
            )

        img_np = img_np.astype(np.float32)
        mask_np = mask_np.astype(np.float32)

        if img_np.max() > 255.0:
            img_np = img_np / 10000.0
        else:
            img_np = img_np / 255.0

        #Handle channels
        if img_np.shape[0] == 3:
            nir_pad = np.zeros((1, 512, 512), dtype=np.float32)
            img_np = np.concatenate((img_np, nir_pad), axis=0)
        elif img_np.shape[0] > 4:
            img_np = img_np[:4, :, :]

        if mask_np.shape[0] > 1:
            mask_np = mask_np[0:1, :, :]
        mask_np = (mask_np > 0.0).astype(np.float32)

        return torch.tensor(img_np), torch.tensor(mask_np)

#train
def train():
    print("Downloading Kaggle Dataset")
    dataset_path = kagglehub.dataset_download("akhilchibber/deforestation-detection-dataset")

    pytorch_dataset = DeforestationDataset(dataset_path)

    if len(pytorch_dataset) == 0:
        print("Dataset is empty debug why")
        return

    dataloader = DataLoader(pytorch_dataset, batch_size=8, shuffle=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = forestClassifier().to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    epochs = 5
    print(f"starting training on {device}...")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for batch_idx, (images, masks) in enumerate(dataloader):
            images = images.to(device)
            masks = masks.to(device)

            predictions = model(images)
            loss = criterion(predictions, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(dataloader)}] | Loss: {loss.item():.4f}")

        avg_loss = epoch_loss / len(dataloader)
        print(f"--- Epoch {epoch+1} Complete | Average Loss: {avg_loss:.4f} ---")

    print("Saving weights")
    torch.save(model.state_dict(), 'canopywatch_v1.pth')

if __name__ == "__main__":
    train()

In [ ]:
#porbally more netowrok styuff + infrence with model/
#